In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 9})
mpl.rcParams['text.latex.preamble']= (
    r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{times}"
)
figurewidth_pt = 433.62
figurewidth_in = figurewidth_pt / 72.27

print(figurewidth_in)

In [ ]:
import jax
# jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", False)
import jax.numpy as jnp
import equinox as eqx 

In [ ]:
from admpe.utils.helper import get_plot_labels
from admpe.data_management import (
    list_excitation_results,
    provide_excitation_results_overview,
    load_excitation_result,
)

from admpe.evaluation.excitation_evaluation import (
    load_excitation_comparison,
    load_corresponding_modeling_results,
)
from admpe.evaluation.excitation_visualization import (
    plot_excitation_feature_grid,
    compare_excitation_feature_grids,
    shared_density_levels,
)
from admpe.evaluation.modeling_evaluation import load_testing_sequences
from admpe.evaluation.shared import visualize_rollout_on_density

In [ ]:
from admpe.utils.helper import pprint_helper
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.data_management import (
    list_modeling_results, load_modeling_result, provide_modeling_results_overview, load_excitation_result
)
from admpe.evaluation.modeling_evaluation import (
    evaluate_model, update_modeling_df, load_corresponding_excitation_result,
    evaluate_model_prediction, evaluate_model_rollout, evaluate_persistence_prediction
)
from admpe.evaluation.modeling_visualization import visualize_model_prediction, visualize_model_rollouts

In [ ]:
SYSTEM = "pendulum" #"pmsm_nominal" # "cart_pole" # "pendulum"
EXPERIMENT = "submission_nov" # "state_only_ses_higher_opt"# "state_only_pm_no_noise" # "state_only_testing_a7percent"
MODELING_EXPERIMENT = "submission_nov_test1" # "state_only_pm_no_noise_gru_testing_validation"
EXCITATION_MODEL = "perfect"
MODEL_TYPE = "gru"
EXCITATION_SEED = 34
N_CONTOUR_LEVELS = 20
CONTOUR_LEVELS_PER_PAIR = True  

OBJECTIVES = (
    "dmpe_uniform",
    "dmpe_static_dynamics",
    "dmpe_static_mixture_dynamics",
    "dmpe_static_curv",
    "dmpe_static_mixture_curv",
    # "dmpe_static_LOLA_ls",
)

LABELS = get_plot_labels(SYSTEM, long=False)

In [ ]:
entries = list_excitation_results(
    system_name=SYSTEM,
    exp_name=EXPERIMENT,
    model_type=EXCITATION_MODEL,
    excitation_objective=list(OBJECTIVES),
    status="completed",
)
if not entries:
    raise ValueError("No completed runs match the configuration.")

# entries = [entries[0]]

excitation_results_overview = provide_excitation_results_overview(entries)

display(
    pd.crosstab(
        excitation_results_overview["objective"], excitation_results_overview["seed"]
    ).reindex(OBJECTIVES, fill_value=0)
)

In [ ]:
runs = load_excitation_comparison(
    excitation_results_overview,
    objectives=OBJECTIVES,
    seed=EXCITATION_SEED,
    dtype=jnp.float32,
)


if SYSTEM == "fluid_tank" and not runs[list(runs.keys())[0]]["metadata"]["params"]["alg_params"]["consider_action_distribution"]:
    CONTOUR_LEVELS = None
    TARGET_CONTOUR_LEVELS = None
else:
    CONTOUR_LEVELS = shared_density_levels(
        runs,
        view="achieved",
        n_levels=N_CONTOUR_LEVELS,
        per_pair=CONTOUR_LEVELS_PER_PAIR,
    )
    
    TARGET_CONTOUR_LEVELS = shared_density_levels(
        runs,
        view="target",
        n_levels=N_CONTOUR_LEVELS,
        per_pair=CONTOUR_LEVELS_PER_PAIR,
    )

display(
    excitation_results_overview.loc[
        excitation_results_overview["seed"].eq(EXCITATION_SEED),
        [
            "objective", "seed", "name", "n_time_steps",
            "bandwidth", "points_per_dim", "elapsed_seconds",
        ],
    ]
)

# TODO:

- show target plots
- show coverage plots

In [ ]:
from itertools import combinations
from matplotlib.ticker import MaxNLocator, ScalarFormatter
from admpe.evaluation.excitation_visualization import plot_excitation_feature_grid

In [ ]:
TARGET_OBJECTIVE_MAP = {
    "dmpe_uniform": r"$\bm{p}^*_\mathrm{U}$",
    "dmpe_static_dynamics": r"$\bm{p}^*_\mathrm{dyn}$",
    "dmpe_static_mixture_dynamics": r"$\bm{p}^*_\mathrm{mix,dyn}$",
    "dmpe_static_curv": r"$\bm{p}^*_\mathrm{curv}$",
    "dmpe_static_mixture_curv": r"$\bm{p}^*_\mathrm{mix,curv}$",
    #"dmpe_static_LOLA_ls": r"$\bm{p}^*_\mathrm{LOLA}$",
}

ACHIEVED_OBJECTIVE_MAP = {
    "dmpe_uniform": r"$\hat{\bm{p}}_\mathrm{U}$",
    "dmpe_static_dynamics": r"$\hat{\bm{p}}_\mathrm{dyn}$",
    "dmpe_static_mixture_dynamics": r"$\hat{\bm{p}}_\mathrm{mix,dyn}$",
    "dmpe_static_curv": r"$\hat{\bm{p}}_\mathrm{curv}$",
    "dmpe_static_mixture_curv": r"$\hat{\bm{p}}_\mathrm{mix,curv}$",
    #"dmpe_static_LOLA_ls": r"$\hat{\bm{p}}_\mathrm{LOLA}$",
}

In [ ]:
def compare_excitation_feature_grids(
    runs,
    *,
    view="achieved",
    labels=None,
    rows_per_figure=12,
    title=None,
    levels=None,
):
    """Compare feature planes with optional shared contour boundaries."""
    if not runs:
        raise ValueError("No runs selected.")
    if view not in {"samples", "target", "achieved"}:
        raise ValueError("Unknown view.")
    if rows_per_figure < 1:
        raise ValueError("rows_per_figure must be positive.")

    def feature_count(run):
        if view == "samples":
            return run["observations"].shape[-1] + run["actions"].shape[-1]
        return run["estimator"].dim

    n = feature_count(next(iter(runs.values())))
    if n < 2 or any(feature_count(run) != n for run in runs.values()):
        raise ValueError("Runs must have the same feature count, at least two.")

    labels = list(range(n)) if labels is None else labels
    pairs = list(combinations(range(n), 2))

    for start in range(0, len(pairs), rows_per_figure):
        page = pairs[start : start + rows_per_figure]

        fig, axes = plt.subplots(
            len(page),
            len(runs),
            figsize=(figurewidth_in, 0.225*figurewidth_in),
            sharex="row",
            sharey="row",
            squeeze=False,
            layout="constrained",
        )

        for column, (objective, run) in enumerate(runs.items()):
            # Map the existing feature-grid positions onto
            # this figure's comparison rows.
            grid = np.full((n, n), None, dtype=object)
            for row, (i, j) in enumerate(page):
                grid[j, i] = axes[row, column]

            plot_excitation_feature_grid(
                run,
                view=view,
                labels=labels,
                fig=fig,
                axs=grid,
                levels=levels,
            )
            if view == "target":
                # element_title = objective.replace("dmpe_", "").replace("_", " ").replace("curv", "curvature") + " " + TARGET_OBJECTIVE_MAP[objective]
                element_title = TARGET_OBJECTIVE_MAP[objective]
            else:
                # element_title = objective.replace("dmpe_", "").replace("_", " ").replace("curv", "curvature") + " " + ACHIEVED_OBJECTIVE_MAP[objective]
                element_title = ACHIEVED_OBJECTIVE_MAP[objective]
            
            axes[0, column].set_title(element_title)

        for row, (i, j) in enumerate(page):
            axes[row, 0].set_ylabel(str(labels[j]))
            for ax in axes[row]:
                ax.set_xlabel(str(labels[i]))

        # fig.suptitle(f"{title or view}")
        for ax in axes[:, 1:].flat:
            ax.set_ylabel("")

        if view in {"target", "achieved"}:
            formatter = ScalarFormatter(useMathText=True, useOffset=False)
            formatter.set_powerlimits((0, 0))
        
            cbar = fig.colorbar(
                axes[0, 0].collections[0],
                ax=axes.ravel().tolist(),
                location="right",
                ticks=MaxNLocator(nbins=3),
                format=formatter,
                aspect=8,
                pad=0.01,
                label="Probability mass \n per grid cell",
            )
            cbar.minorticks_off()
            offset = cbar.ax.yaxis.get_offset_text()
            offset.set_x(0.5)
            offset.set_horizontalalignment("center")
            
            yield fig, axes


In [ ]:
VIEW = "target"

for fig, axes in compare_excitation_feature_grids(
    runs,
    view=VIEW,
    title=r"Target distribution $\bm{p}^*$",
    labels=LABELS["both"],
    levels=TARGET_CONTOUR_LEVELS if VIEW == "target" else CONTOUR_LEVELS,
):
    display(fig)
    plt.savefig(f"paper_fig/exemplary_pendulum_{VIEW}_distributions.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"paper_fig/exemplary_pendulum_{VIEW}_distributions.svg", bbox_inches="tight")
    plt.savefig(f"paper_fig/exemplary_pendulum_{VIEW}_distributions.pdf", bbox_inches="tight")
    plt.close(fig)

In [ ]:
VIEW = "achieved"

for fig, axes in compare_excitation_feature_grids(
    runs,
    view=VIEW,
    title=r"Achieved data distribution $\hat{\bm{p}}$",
    labels=LABELS["both"],
    levels=TARGET_CONTOUR_LEVELS if VIEW == "target" else CONTOUR_LEVELS,
):
    display(fig)
    plt.savefig(f"paper_fig/exemplary_pendulum_{VIEW}_distributions.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"paper_fig/exemplary_pendulum_{VIEW}_distributions.svg", bbox_inches="tight")
    plt.savefig(f"paper_fig/exemplary_pendulum_{VIEW}_distributions.pdf", bbox_inches="tight")
    plt.close(fig)